# TB Portals — **Agentic 6-Rung Pipeline (headline run)**: mode `a2`
Caches RAD-DINO CLS+patch-grid features, runs rungs `1 2 3 5 6 7` + agentic_best variants + spatial-cavity probe, **5 seeds × M=10**, saves heads. Download `agentic_a2.zip`.
Attach only **tb-portals-cxr-pngs**; Internet **ON**; GPU T4.

## 0 — Clone  *(restart kernel after any pull that changed .py)*

In [1]:
import os, sys, subprocess
REPO_URL = "https://github.com/mabdullahi7780/dl-project-codebase.git"
REPO_DIR = "/kaggle/working/dl-project-codebase"
BRANCH   = "cleaned-repo"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
for _p in (REPO_DIR, REPO_DIR + "/scripts"):
    if _p not in sys.path: sys.path.insert(0, _p)
print("repo ready at", REPO_DIR)

Cloning into '/kaggle/working/dl-project-codebase'...


repo ready at /kaggle/working/dl-project-codebase


Updating files: 100% (465/465), done.


## Install deps

In [2]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "torchxrayvision", "pydicom", "pylibjpeg", "pylibjpeg-libjpeg"], check=False)
print("deps installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.0/29.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 74.8 MB/s eta 0:00:00
deps installed


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

## Paths

In [3]:
import os
WORK          = "/kaggle/working"
REPO_DIR      = "/kaggle/working/dl-project-codebase"
DATASET       = "/kaggle/input/datasets/mabdullahi454/tb-portals-cxr-pngs"
KAGGLE_EXPORT = f"{DATASET}/kaggle_export"
PAPER_MANIFEST = f"{WORK}/tbportals_manifest_paper.csv"
MODE          = "a2"
BACKBONE      = "rad-dino"
FEATURES      = f"{WORK}/features_{BACKBONE}_cls.npz"
FEATURES_GRID = f"{WORK}/features_{BACKBONE}_grid7.npz"
print("aux grid features ->", FEATURES_GRID)
print("MODE:", MODE, "| KAGGLE_EXPORT:", os.path.isdir(KAGGLE_EXPORT), "| primary features ->", FEATURES)

aux grid features -> /kaggle/working/features_rad-dino_grid7.npz
MODE: a2 | KAGGLE_EXPORT: True | primary features -> /kaggle/working/features_rad-dino_cls.npz


## 1 — Build the 5,010-image manifest (Kantipudi Table 1)

In [4]:
import sys, pandas as pd
from pathlib import Path
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
from build_paper_manifest import subsample, PAPER_TOTAL
raw = pd.read_csv(f"{KAGGLE_EXPORT}/manifest.csv",
                  dtype={"image_id": str, "patient_id": str, "country": str})
raw["image_path"] = raw["image_path"].apply(
    lambda p: p if str(p).startswith("/") else f"{KAGGLE_EXPORT}/{p}")
paper_df = subsample(raw, seed=42)
paper_df["image_id"] = paper_df["image_path"].apply(lambda p: Path(str(p)).stem)
paper_df.to_csv(PAPER_MANIFEST, index=False)
print(f"Paper manifest: {len(paper_df)} images (target {PAPER_TOTAL}) -> {PAPER_MANIFEST}")

country       need_c  have_c  need_n  have_n
Georgia          713    1298     701    1108
Belarus          254     285     798     894
Ukraine          500    1206     816    1775
Kazakhstan       159     304     240     384
Romania          143     233      77     171
Moldova          193     278     396     527
Azerbaijan         5       7      12      18
India              0       6       3      12
Paper manifest: 5010 images (target 5010) -> /kaggle/working/tbportals_manifest_paper.csv


## 2 — Cache primary features (one-time; idempotent)

In [5]:
import os, sys
if REPO_DIR + "/scripts" not in sys.path: sys.path.insert(0, REPO_DIR + "/scripts")
if os.path.isfile(FEATURES):
    print("primary features already cached ->", FEATURES)
else:
    from cache_features import main as cache_main
    cache_main(["--manifest", PAPER_MANIFEST, "--out", FEATURES,
                "--backbone", BACKBONE, "--batch-size", "32"])

[cache] backbone=rad-dino device=cuda patch_grid=off (CLS)


preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[cache] feature dim = 768
[cache] 320/5010
[cache] 640/5010
[cache] 960/5010
[cache] 1280/5010
[cache] 1600/5010
[cache] 1920/5010
[cache] 2240/5010
[cache] 2560/5010
[cache] 2880/5010
[cache] 3200/5010
[cache] 3520/5010
[cache] 3840/5010
[cache] 4160/5010
[cache] 4480/5010
[cache] 4800/5010
[cache] wrote features (5010, 768) -> /kaggle/working/features_rad-dino_cls.npz


## 2b — Cache patch-grid features (for the spatial cavity head)

In [6]:
if os.path.isfile(FEATURES_GRID):
    print("grid features already cached ->", FEATURES_GRID)
else:
    from cache_features import main as cache_main
    cache_main(["--manifest", PAPER_MANIFEST, "--out", FEATURES_GRID,
                "--backbone", BACKBONE, "--patch-grid", "7", "--batch-size", "32"])

[cache] backbone=rad-dino device=cuda patch_grid=7


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

[cache] feature dim = 768
[cache] 320/5010
[cache] 640/5010
[cache] 960/5010
[cache] 1280/5010
[cache] 1600/5010
[cache] 1920/5010
[cache] 2240/5010
[cache] 2560/5010
[cache] 2880/5010
[cache] 3200/5010
[cache] 3520/5010
[cache] 3840/5010
[cache] 4160/5010
[cache] 4480/5010
[cache] 4800/5010
[cache] wrote features (5010, 49, 768) -> /kaggle/working/features_rad-dino_grid7.npz


## 3 — Run the rung ladder for mode `a2` (rungs 1 2 3 5 6 7, 5 seeds, M=10)

In [7]:
from src.training.train_agentic import main as agentic_main
args = ["--features", FEATURES, "--manifest", PAPER_MANIFEST,
        "--mode", MODE, "--out-dir", f"{WORK}/agentic_{MODE}",
        "--rungs", '1', '2', '3', '5', '6', '7',
        "--seeds", '0', '1', '2', '3', '4',
        "--ensemble-m", "10",
        "--held-outs", "Romania", "Moldova", "Kazakhstan",
        "--save-heads"]
args += ["--features-grid", FEATURES_GRID]
agentic_main(args)

[agentic] aux grid cache loaded: dim=768 P=49
[agentic] device=cuda mode=a2 dim=768 feat=CLS grid_available=True rungs=[1, 2, 3, 5, 6, 7] seeds=[0, 1, 2, 3, 4] M=10
[agentic] 15 configs: ['rung1_mse', 'rung1_bmc', 'rung2_tta', 'rung3_retrieval', 'rung5_moe', 'rung6_conformal', 'rung7_calibrate_lin', 'rung7_calibrate_iso', 'agentic_best', 'agentic_v2_lin', 'agentic_v2_iso', 'agentic_best_tta', 'rung4b_spatial_cavity', 'agentic_best_spatcav', 'stacked_legacy']
[tbportals] split held_out=Romania: train=3832 val=958 test=220 (train/val patients 3618/904).
[a2|rung1_mse] Romania s0  MAE=19.65 CI[17.3,22.1] r=0.652 a=0.00 | base 20.11 paper 18.70 -> within-noise
[tbportals] split held_out=Romania: train=3836 val=954 test=220 (train/val patients 3618/904).
[a2|rung1_mse] Romania s1  MAE=22.19 CI[19.7,24.6] r=0.608 a=0.00 | base 20.11 paper 18.70 -> within-noise
[tbportals] split held_out=Romania: train=3843 val=947 test=220 (train/val patients 3618/904).
[a2|rung1_mse] Romania s2  MAE=21.33 C

## 4 — Save (download → baseline_runs/agentic_runs/)

In [8]:
import os, shutil
src = f"{WORK}/agentic_{MODE}"
try: shutil.copy(FEATURES, f"{src}/{os.path.basename(FEATURES)}")
except Exception as e: print("primary cache copy skipped:", e)
try: shutil.copy(FEATURES_GRID, f"{src}/{os.path.basename(FEATURES_GRID)}")
except Exception as e: print("grid cache copy skipped:", e)
zip_path = shutil.make_archive(f"{WORK}/agentic_{MODE}", "zip", src)
print("Saved ->", zip_path)
print("Contents include: results_agentic_*.csv, preds_*.csv, heads/*.pt, features_*.npz")

Saved -> /kaggle/working/agentic_a2.zip
Contents include: results_agentic_*.csv, preds_*.csv, heads/*.pt, features_*.npz
